# Step 2-5. (심화) Distillation 준비: 스승의 판단을 모으기
**Distillation(증류)** 은 큰 모델(스승)의 답을 작은 모델(제자)이 따라 하도록 가르치는 방법입니다. 비슷한 일을 훨씬 싸고 빠르게 하려는 것이 목적입니다.

Step 2-4에서 한 번에 한 증류를 두 노트북(2-5, 2-6)으로 나눠 한 단계씩 봅니다. 증류할 일은 Step 2-2 신약 팀의 **라우팅**입니다. "이 질문은 어느 전문가에게 보낼까?"

## 순서
1. 질문을 많이 만든다 (이 노트북)
2. 스승 모델에게 질문마다 답을 달게 한다 (이 노트북)
3. 그 질문과 답으로 제자 모델을 학습시킨다 (Step 2-6)

## 핵심 개념
- **Teacher(스승)**: 큰 모델. 여기서는 `gemma4:e4b`
- **Student(제자)**: 작은 모델. Step 2-6에서 10분의 1 크기 모델을 씁니다
- **Label(라벨)**: 정답. 예전에는 사람이 달았지만 여기서는 스승 모델이 답니다

In [ ]:
# 필요한 패키지를 설치합니다. 처음에는 1~2분 걸립니다.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "ollama"], check=True)
print("✅ 패키지 준비 완료. 다음 셀로 넘어가세요.")

## 1. 설정

In [ ]:
import os, sys
import ollama


EXPERT_MODEL = "gemma4:e4b"    # 전문가와 지휘자 역할을 하는 모델 (약 9.5GB)
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")   # GPU 서버를 쓰면 그 주소로 바뀝니다

installed = [m.model for m in ollama.list().models]
if not any(m == EXPERT_MODEL or m.startswith(EXPERT_MODEL + ":") for m in installed):
    print(f"⬇️  {EXPERT_MODEL} 내려받는 중... (약 9.5GB, 처음 한 번만)")
    ollama.pull(EXPERT_MODEL)
print(f"✅ 모델 준비 완료: {EXPERT_MODEL}  (Ollama 주소: {OLLAMA_HOST})")

## 2. 질문 만들기
질문 틀 36개에 약 이름 12개(와 약 이름 없이 묻는 경우)를 조합해 질문 약 500개를 만듭니다. 실제 서비스라면 **사용자가 실제로 한 질문 기록**을 쓰는 것이 가장 좋습니다.

질문 틀 가운데 6개는 따로 빼 둡니다. 제자가 **학습 때 본 적 없는 종류의 질문**에도 맞게 답하는지 확인하기 위해서입니다.

In [ ]:
import itertools, json, random

DRUGS = ["아스피린", "이부프로펜", "메트포르민", "아토르바스타틴", "와파린", "오메프라졸",
         "암로디핀", "세마글루타이드", "이 화합물", "이 후보물질", "우리 신약 후보", "화합물 A",
         "", ""]      # 빈 칸은 약 이름 없이 질문만 하는 경우

# TODO: 질문 틀을 추가해 보세요! {d} 자리에 약 이름이 들어갑니다.
TRAIN_TEMPLATES = [
    "{d}: 용해도를 높이려면 구조를 어떻게 바꿔야 해?", "{d}: 구조가 비슷한 화합물을 찾아 줘", "{d}: Lipinski 규칙을 만족해?",
    "{d}: 분자량과 logP가 어떻게 돼?", "{d}: 독성을 일으킬 만한 구조가 있어?", "{d}: 합성이 어려운 부분은 어디야?",
    "{d}: 심장독성 위험은?", "{d}: 반감기와 하루 복용 횟수는?", "{d}: 신장이 나쁜 환자에게 써도 돼?",
    "{d}: 같이 먹으면 안 되는 약은?", "{d}: 흔한 부작용은 뭐야?", "{d}: 간에서 어떻게 대사돼?",
    "{d}: FDA 승인 가능성은?", "{d}: 비슷한 약의 임상시험 현황은?", "{d}: 임상 2상을 어떻게 설계할까?",
    "{d}: 허가 전략을 세워 줘", "{d}: 시장에서 경쟁력이 있을까?", "{d}: 비슷한 약이 임상에서 실패한 사례가 있어?",
    "{d}: 다음 단계로 진행해도 될까?", "{d}: 종합 평가해 줘", "{d}: 구조를 바꾸면 부작용과 허가에 어떤 영향이 있어?",
    "{d}: 안전성 문제로 임상이 중단된 적 있어?", "{d}: 독성을 줄이면서 효과를 유지하는 방법은?", "{d}: 용량을 올리면 승인에 문제가 될까?",
    "{d}: 고령 환자에게 용량을 어떻게 조절해?", "{d}: 수용성이 낮은 이유가 뭐야?", "{d}: 경쟁 약보다 나은 점을 임상에서 어떻게 보일까?",
    "{d}: 임산부에게 안전해?", "{d}: 특허를 피할 수 있는 구조 변경은?", "{d}: 희귀의약품 지정을 받을 수 있을까?",
]
TEST_TEMPLATES = [   # 학습에 쓰지 않는 질문 틀
    "{d}: 먹는 약으로 만들 수 있는 구조야?", "{d}: 과다 복용하면 어떻게 돼?", "{d}: 임상 3상까지 얼마나 걸릴까?",
    "{d}: 흡수가 잘 안 되는데 구조 문제야, 제형 문제야?", "{d}: 소아 임상을 해야 할까?", "{d}: Go인지 No-Go인지 판단해 줘",
]

def fill(template, drug):
    return template.format(d=drug).removeprefix(": ")


random.seed(7)
train_q = [fill(t, d) for t, d in itertools.product(TRAIN_TEMPLATES, DRUGS)]
test_q = [fill(t, d) for t, d in itertools.product(TEST_TEMPLATES, DRUGS[:-1])]
random.shuffle(train_q)
random.shuffle(test_q)
print(f"✅ 학습용 질문 {len(train_q)}개, 처음 보는 유형의 시험용 질문 {len(test_q)}개")
print("   예:", train_q[0])

## 3. 스승에게 답을 달게 하기
Step 2-2의 지휘자와 **똑같은 지시문**을 씁니다. 약 500개에 2~3분 걸립니다.

In [ ]:
ROUTER_PROMPT = """너는 신약 개발팀의 지휘자다. 질문에 답하는 데 꼭 필요한 전문가만 고른다.
- chemist: 화합물 구조, 물리화학적 특성, 구조 변경, 유사 화합물
- pharmacologist: 약동학, 약력학, 독성, 부작용, 약물 상호작용, 용량
- regulatory: 임상시험 현황과 설계, 허가 전략, 시장과 경쟁
질문 하나에 1명에서 3명까지 고를 수 있다."""
ROUTER_SCHEMA = {"type": "object", "required": ["experts"], "properties": {"experts": {
    "type": "array", "minItems": 1, "maxItems": 3,
    "items": {"type": "string", "enum": ["chemist", "pharmacologist", "regulatory"]}}}}
ORDER = ["chemist", "pharmacologist", "regulatory"]


def route_with_teacher(question):
    """큰 모델(지휘자)이 필요한 전문가를 고른다. 답은 정해진 형식(JSON)으로만 받는다."""
    kw = dict(model=EXPERT_MODEL, format=ROUTER_SCHEMA, options={"temperature": 0},
              messages=[{"role": "system", "content": ROUTER_PROMPT}, {"role": "user", "content": question}])
    try:
        r = ollama.chat(think=False, **kw)     # 생각 과정을 끄면 빨라집니다
    except Exception:
        r = ollama.chat(**kw)
    return sorted(set(json.loads(r["message"]["content"])["experts"]), key=ORDER.index)

import time


def label_all(questions, name):
    rows, start = [], time.time()
    for i, q in enumerate(questions, 1):
        rows.append({"question": q, "experts": ",".join(route_with_teacher(q))})
        if i % 60 == 0 or i == len(questions):
            print(f"   {name}: {i}/{len(questions)}  ({time.time() - start:.0f}초)")
    return rows


train_rows = label_all(train_q, "학습용")
test_rows = label_all(test_q, "시험용")
print("✅ 스승의 답 모으기 완료")

## 4. 스승의 답 살펴보기
제자는 스승을 따라 배우므로 **스승이 틀리면 제자도 틀립니다.** 눈으로 확인합니다.

In [ ]:
from collections import Counter

print("전문가 조합별 개수:")
for label, n in Counter(r["experts"] for r in train_rows).most_common():
    print(f"   {n:>4}  {label}")

print("\n몇 개만 직접 보기:")
for r in random.sample(train_rows, 10):
    print(f"   {r['experts']:<36} {r['question']}")

## 5. 학습용 파일로 저장
학습 도구가 읽는 형식(JSONL, 한 줄에 대화 하나)으로 저장합니다. 제자에게는 **짧은 지시문**만 줍니다. 긴 설명 없이도 답하도록 가르치는 것이 목표입니다.

In [ ]:
INSTRUCTION = "질문에 답하는 데 필요한 전문가를 chemist, pharmacologist, regulatory 중에서 골라 쉼표로만 답하라.\n질문: "


def to_chat(row, with_answer=True):
    messages = [{"role": "user", "content": INSTRUCTION + row["question"]}]
    if with_answer:
        messages.append({"role": "assistant", "content": row["experts"]})
    return {"messages": messages}


os.makedirs("distill/data", exist_ok=True)
n_valid = 40
splits = {"train": train_rows[n_valid:], "valid": train_rows[:n_valid], "test": test_rows}
for name, rows in splits.items():
    with open(f"distill/data/{name}.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(to_chat(r), ensure_ascii=False) + "\n")
    print(f"✅ distill/data/{name}.jsonl  {len(rows)}건")

json.dump({"instruction": INSTRUCTION, "teacher": EXPERT_MODEL}, open("distill/meta.json", "w", encoding="utf-8"), ensure_ascii=False)
print("\n한 줄 예시:")
print(open("distill/data/train.jsonl", encoding="utf-8").readline()[:300])

## 정리
- 증류의 학습 자료는 "질문 + 스승의 답"입니다. 사람이 라벨을 달지 않았습니다.
- 자료의 품질이 제자의 품질을 정합니다. 그래서 스승의 답을 눈으로 확인했습니다.
- 다음은 이 자료로 작은 모델을 학습시킵니다. → Step 2-6